# **Set-up**

In [ ]:
!pip install --pre deepchem
import deepchem
deepchem.__version__

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 608 kB 32.7 MB/s 
     |████████████████████████████████| 22.7 MB 1.4 MB/s 


'2.6.1'

In [ ]:
!pip install rdkit

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 22.7 MB 1.3 MB/s 


In [ ]:
import deepchem as dc


In [ ]:
import ast

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras

import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import BondType
from rdkit.Chem.Draw import MolsToGridImage

RDLogger.DisableLog("rdApp.*")

# **Load and Check Data**

In [ ]:
# load data
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving MolGANInput.csv to MolGANInput.csv
User uploaded file "MolGANInput.csv" with length 60041 bytes


In [ ]:
df = pd.read_csv('PurchasableFragments.csv')
df = df.rename(columns={'Fragment': 'smiles'})
df.head()
del df['Unnamed: 0']

In [ ]:
data_smiles = df.to_numpy()
# --- processing data to array, then nparray for training ---
data_array = []

#iterate through data_smiles and add smiles to data_array
for i in range(0, len(data_smiles)):
  data_array.append(data_smiles[i][0])

#convert data_array (previously data_smiles) to numpy array
import numpy as np
train_smiles = np.asarray(data_array)
print(type(train_smiles))

#print 1st SMILES string just to see SMILES string format
print(train_smiles[0])

<class 'numpy.ndarray'>
Cn1ccccc1=NC1CC1


# **Train Model**

In [ ]:
tokens = set()
for s in train_smiles:
  tokens = tokens.union(set(s))
tokens = sorted(list(tokens))
max_length = max(len(s) for s in train_smiles)
import tensorflow as tf


In [ ]:
# --- The model is a VAE model from deepchem ---
#the encoder = CNN and the decoder = GRU
from deepchem.models.optimizers import Adam, ExponentialDecay
from deepchem.models.seqtoseq import AspuruGuzikAutoEncoder

learning_rate = ExponentialDecay(0.001, 0.95, 22.6)

#declaring model which is AspuruGuzikAutoEncoder, pre-trained model in deepchem library
model = AspuruGuzikAutoEncoder(tokens, max_length, model_dir='vae',optimizer=Adam(learning_rate=learning_rate))
batches_per_epoch = len(train_smiles)/model.batch_size
print('BATCHES PER EPOCH')
print(batches_per_epoch)

#Define the function for generating molecules
def generate_sequences(epochs):
  for i in range(epochs):
    for s in train_smiles:
      yield (s, s)

#deepchem has its own fit model variation - 1000 epochs tends to have the model work better
model.fit_sequences(generate_sequences(1000))



BATCHES PER EPOCH
22.6


# **Generate Molecules**

In [ ]:
#First number is the number of molecules that you want to generate

predictions = model.predict_from_embeddings(np.random.normal(size=(40,196)))
molecules = []

In [ ]:
#using chem from rdkit to ensure generated molecules are valid
total_molecules = 0
for p in predictions:
  total_molecules = total_molecules + 1
  smiles = ''.join(p)
  if Chem.MolFromSmiles(smiles) is not None:
    molecules.append(smiles)

for m in molecules:
  print(m)

CCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCC
CC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCC
CCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCC
CCC
CCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCC
CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCO
CCCCCCCCCCCCCCCCCCCCC


In [ ]:
#Get the data output to match that of other modes
moleculesdf = pd.DataFrame(molecules)
renamed_molecules = moleculesdf.rename(columns={0: 'smiles'})

In [ ]:
renamed_molecules

,smiles
0,CCCCCCCCCCCCCCC
1,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
2,CCCCCCCCCCCCCCCCCCCCCCC
3,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
4,CCCCCCCCCCCCCCC
5,CC
6,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
7,CCCCCCCCCCCCC
8,CCCCCCCCCCC
9,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC


In [ ]:
renamed_molecules.sort_values(by='smiles')

,smiles
5,CC
13,CCC
8,CCCCCCCCCCC
7,CCCCCCCCCCCCC
0,CCCCCCCCCCCCCCC
4,CCCCCCCCCCCCCCC
11,CCCCCCCCCCCCCCCCC
12,CCCCCCCCCCCCCCCCCC
17,CCCCCCCCCCCCCCCCCCC
10,CCCCCCCCCCCCCCCCCCCCC


In [ ]:
unique_molecules = renamed_molecules.drop_duplicates()

In [ ]:
unique_molecules

,smiles
0,CCCCCCCCCCCCCCC
1,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
2,CCCCCCCCCCCCCCCCCCCCCCC
5,CC
7,CCCCCCCCCCCCC
8,CCCCCCCCCCC
9,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
10,CCCCCCCCCCCCCCCCCCCCC
11,CCCCCCCCCCCCCCCCC
12,CCCCCCCCCCCCCCCCCC


In [ ]:
print('Unique Molecules Percentage to total molecules is: ' + str(100 * (unique_molecules.size/total_molecules)) + '%')
print('Unique&Valid Molecules to Valid Molecules is: ' + str(100 * (unique_molecules.size/renamed_molecules.size))+ '%')
print('Ratio U&V to V is ' + str(unique_molecules.size) + '/' + str(renamed_molecules.size))
print('Ratio U&V to total is ' + str(unique_molecules.size) + '/' + str(total_molecules))

Unique Molecules Percentage to total molecules is: 40.0%
Unique&Valid Molecules to Valid Molecules is: 80.0%
Ratio U&V to V is 16/20
Ratio U&V to total is 16/40


In [ ]:
from google.colab import files
unique_molecules.to_csv('VAE_sample3.csv')
files.download('VAE_sample3.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>